In [2]:
from bot.config import load_config
from bot.market_state import get_timestamp, get_market_slug, get_window_label
from bot.markets import get_market_by_slug
from bot.feeds.chainlink import ChainlinkFeed
from bot.state.polymarket_order_book import PolymarketOrderBook
from bot.feeds.polymarket_state import PolymarketState
from bot.feeds.polymarket_crypto_price import PolymarketCryptoPrice
from bot.client import get_client
from py_clob_client_v2.order_builder.constants import BUY
from py_clob_client_v2 import ClobClient, OrderArgs, PartialCreateOrderOptions, MarketOrderArgs, OrderType
from eth_account import Account
from eth_account.messages import encode_defunct
from bot.predict.client import PredictClient, get_minimum_order_size

import requests
import websockets
import time
import json
import asyncio
import httpx
import base64

In [3]:
proxies = {
    "http": "socks5h://127.0.0.1:9090",
    "https": "socks5h://127.0.0.1:9090",
}

# REST API — works through SOCKS5
r = requests.get("https://api.binance.com/api/v3/ticker/price",
                  params={"symbol": "BTCUSDT"}, proxies=proxies)
print(r.json())

{'symbol': 'BTCUSDT', 'price': '77408.08000000'}


In [4]:
config = load_config()
slug = get_market_slug(config.crypto, config.interval_minutes)
market = await get_market_by_slug(slug)

In [5]:
market

Market(condition_id='0x1b910e16e8ef040b85a1ba8e73c4acc2c6e9281554d688a1de96057de19bd554', question_id='0x5d6adfd9375f8a94dabc1db968442763ea47f22876bc0992efaf06fd975e7403', question='Bitcoin Up or Down - May 21, 9:05PM-9:10PM ET', slug='btc-updown-5m-1779411900', up_token_id='48152694129698459320606682092454602099426373197019858344998816218667073179034', down_token_id='87579509526667738715027991099565354411415039768582464446685130460133775605583', outcomes=['Up', 'Down'], outcome_prices=['0.505', '0.495'], tick_size='0.01', neg_risk=False, active=True, end_date='2026-05-22T01:10:00Z', description='This market will resolve to "Up" if the Bitcoin price at the end of the time range specified in the title is greater than or equal to the price at the beginning of that range. Otherwise, it will resolve to "Down".\nThe resolution source for this market is information from Chainlink, specifically the BTC/USD data stream available at https://data.chain.link/streams/btc-usd.\nPlease note that thi

In [6]:
market.up_token_id

'48152694129698459320606682092454602099426373197019858344998816218667073179034'

In [7]:
client = get_client()

In [8]:
# order = client.create_and_post_market_order(
#     order_args=MarketOrderArgs(
#         token_id=market.up_token_id,
#         amount=1,
#         side=BUY,
#     ),
#     options=PartialCreateOrderOptions(tick_size="0.01", neg_risk=market.neg_risk),
# )

In [9]:
# order = client.create_and_post_order(
#     order_args=OrderArgs(
#         token_id=market.down_token_id,
#         price=0.01,
#         size=100,
#         side=BUY,
#     ),
#     options=PartialCreateOrderOptions(tick_size="0.01", neg_risk=market.neg_risk),
#     order_type=OrderType.FOK,
# )

In [10]:
predict_config = config.predict

In [11]:
predict_config.api_host

'https://api-testnet.predict.fun'

In [12]:
predict_client = PredictClient(config)

In [13]:
predict_client.get_category_slug()

'btc-updown-5m-1779411900'

In [14]:
market = await predict_client.get_current_5m_crypto_market()

No predict.fun market found for categorySlug=btc-updown-5m-1779411900 (83 markets scanned)


In [15]:
market

True